# Purview CLI Unified Catalog Quality - Sample Notebook

This notebook demonstrates the new `pvw uc quality` commands end-to-end.

## Prerequisites

1. You are authenticated (`az login`) and have Purview access.
2. The `pvw` command is available in your environment.
3. Set the environment values below before running command cells.

In [ ]:
import json
import os
import shlex
import subprocess
from typing import Optional

In [ ]:
# Update these values for your tenant/account before running commands.
os.environ['PURVIEW_ACCOUNT_NAME'] = os.environ.get('PURVIEW_ACCOUNT_NAME', 'your-purview-account-name')
os.environ['PURVIEW_RESOURCE_GROUP'] = os.environ.get('PURVIEW_RESOURCE_GROUP', 'your-resource-group')
# Optional if auto-discovery works in your environment:
# os.environ['PURVIEW_ACCOUNT_ID'] = 'xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx'

print('PURVIEW_ACCOUNT_NAME =', os.environ['PURVIEW_ACCOUNT_NAME'])
print('PURVIEW_RESOURCE_GROUP =', os.environ['PURVIEW_RESOURCE_GROUP'])

In [ ]:
def run_cmd(command: str, expect_json: bool = False) -> Optional[dict]:
    print(f'\n$ {command}')
    result = subprocess.run(
        shlex.split(command),
        capture_output=True,
        text=True,
        env=os.environ.copy(),
        check=False,
    )

    if result.stdout:
        print(result.stdout.strip())
    if result.stderr:
        print('STDERR:', result.stderr.strip())

    if result.returncode != 0:
        print(f'Command failed with exit code {result.returncode}')
        return None

    if expect_json:
        try:
            return json.loads(result.stdout)
        except json.JSONDecodeError:
            print('Output was not valid JSON.')
            return None

    return None

In [ ]:
run_cmd('pvw --version')
run_cmd('pvw uc quality --help')

In [ ]:
domains = run_cmd('pvw uc quality domains --output json', expect_json=True)

domain_id = None
if isinstance(domains, dict) and isinstance(domains.get('value'), list) and domains['value']:
    domain_id = domains['value'][0].get('id')
elif isinstance(domains, list) and domains:
    domain_id = domains[0].get('id')

print('Selected domain_id =', domain_id)

In [ ]:
if domain_id:
    run_cmd(f'pvw uc quality domain-report --domain-id {domain_id} --output json')
    run_cmd(f'pvw uc quality data-sources --domain-id {domain_id} --output json')
    run_cmd(f'pvw uc quality schedules --domain-id {domain_id} --output json')
    run_cmd(f'pvw uc quality alerts --domain-id {domain_id} --output json')
    run_cmd(f'pvw uc quality assets --domain-id {domain_id} --output json')
    run_cmd(f'pvw uc quality score domain --domain-id {domain_id} --output json')
else:
    print('No domain_id found. Check your account context and try again.')

## Next Steps

- Replace `domain_id` with a specific business domain you want to inspect.
- Save command outputs to local files for audit/reporting if needed.
- Integrate these calls into your operational runbooks.